# 회로 양자화 (Circuit Quantization)

이 노트북에서는 YAML 기술로부터 초전도 회로를 양자화하는 전체 파이프라인을 보여줍니다.

**다루는 내용:**
- 예제 1: 단일 Transmon (가장 간단한 회로)
- 예제 2: Transmon + 션트 커패시터
- 예제 3: 대칭 SQUID (자속 가변)
- 예제 4: 비대칭 SQUID
- 예제 5: 하드코딩 vs 회로 검증

In [1]:
using ScQubitsMimic
using CairoMakie

ArgumentError: ArgumentError: Package CairoMakie not found in current path.
- Run `import Pkg; Pkg.add("CairoMakie")` to install the CairoMakie package.

## 회로 양자화 파이프라인

`Circuit`은 YAML 형식의 회로 기술을 입력으로 받아 다음의 단계를 자동으로 수행합니다:

1. **파싱** → `CircuitGraph` (노드, 브랜치, 타입: C, L, JJ, CJ)
2. **토폴로지 분석** → 스패닝 트리, 기본 루프, 클로저 브랜치
3. **기호적 분석** → `SymbolicCircuit` (커패시턴스/인덕턴스 행렬, 기호 해밀토니안)
4. **모드 분해** → 주기적/확장/자유/동결 모드 분류
5. **양자화** → 수치 해밀토니안 구성 및 대각화

### 브랜치 타입

| 타입 | 설명 | 파라미터 |
|------|------|----------|
| `JJ` | 조셉슨 접합 | $E_J$, $E_C$ |
| `C` | 커패시터 | $E_C$ |
| `L` | 인덕터 | $E_L$ |
| `CJ` | 커패시터 + 조셉슨 접합 | $E_J$, $E_C$ |

---
## 예제 1: 단일 Transmon

가장 간단한 초전도 회로: 접지와 노드 1 사이의 단일 조셉슨 접합.

$$H = 4E_C \hat{n}^2 - E_J \cos\hat{\varphi}$$

In [2]:
desc1 = """
branches:
  - [JJ, 0, 1, EJ=30.0, EC=1.2]
"""

circ1 = Circuit(desc1; ncut=30)

println("회로: 접지(0) ↔ 노드(1), 단일 JJ")
println("\n기호적 해밀토니안 (노드 기저):")
println("  H = ", sym_hamiltonian_node(circ1))
println("\n기호적 해밀토니안 (모드 기저):")
println("  H = ", sym_hamiltonian(circ1))

회로: 접지(0) ↔ 노드(1), 단일 JJ

기호적 해밀토니안 (노드 기저):
  H = -30.0cos(φ₁) + 19.2(n₁^2)

기호적 해밀토니안 (모드 기저):
  H = -30.0cos(θ₁) + 19.2(nθ₁^2)


In [3]:
# 변수 변환 및 모드 분류
T, vc = variable_transformation(circ1)
println("변수 변환 행렬 T:")
println("  T = ", T)
println("\n모드 분류:")
println("  주기적 모드: ", vc.periodic)
println("  확장 모드:   ", vc.extended)
println("  외부 자속:   ", external_fluxes(circ1))
println("  오프셋 전하: ", offset_charges(circ1))

변수 변환 행렬 T:
  T = [1.0;;]

모드 분류:
  주기적 모드: [1]
  확장 모드:   Int64[]
  외부 자속:   Symbolics.Num[]
  오프셋 전하: Symbolics.Num[ng₁]


In [4]:
evals1 = eigenvals(circ1; evals_count=4)

println("처음 4개 고유값:")
for (i, e) in enumerate(evals1)
    println("  E_$(i-1) = $(round(e, digits=6)) GHz")
end
println("\n  ω₀₁ = $(round(evals1[2] - evals1[1], digits=6)) GHz")

처음 4개 고유값:
  E_0 = -14.447063 GHz
  E_1 = 15.444879 GHz
  E_2 = 29.520347 GHz
  E_3 = 78.209262 GHz

  ω₀₁ = 29.891942 GHz


---
## 예제 2: Transmon + 션트 커패시터

JJ에 병렬 커패시터를 추가하면 총 커패시턴스가 증가하여 $E_C$가 감소합니다.

$$C_{\text{total}} = C_{JJ} + C_{\text{shunt}}$$

이는 $E_C$를 낮추어 전하 잡음에 대한 민감도를 더 줄이는 일반적인 기법입니다.

In [5]:
desc2 = """
branches:
  - [JJ, 0, 1, EJ=10.0, EC=0.5]
  - [C, 0, 1, EC=1.0]
"""

circ2 = Circuit(desc2; ncut=30)

println("회로: JJ (EJ=10, EC=0.5) + C (EC=1.0) 병렬")
println("\n기호적 해밀토니안:")
println("  ", sym_hamiltonian(circ2))

evals2 = eigenvals(circ2; evals_count=4)
println("\n  ω₀₁ = $(round(evals2[2] - evals2[1], digits=6)) GHz")

회로: JJ (EJ=10, EC=0.5) + C (EC=1.0) 병렬

기호적 해밀토니안:
  -10.0cos(-Φext₁ + θ₁) + 5.333333333333333(nθ₁^2)

  ω₀₁ = 9.069117 GHz


---
## 예제 3: 대칭 SQUID (자속 가변 Transmon)

두 개의 동일한 JJ가 루프를 형성하여 외부 자속으로 유효 $E_J$를 조절할 수 있습니다.

$$E_J^{\text{eff}}(\Phi_{\text{ext}}) = 2E_J \left|\cos\left(\frac{\Phi_{\text{ext}}}{2}\right)\right|$$

여기서 $\Phi_{\text{ext}}$는 루프를 관통하는 외부 자속입니다.

In [6]:
desc3 = """
branches:
  - [JJ, 0, 1, EJ=10.0, EC=0.3]
  - [JJ, 0, 1, EJ=10.0, EC=0.3]
  - [C, 0, 1, EC=0.5]
"""

circ3 = Circuit(desc3; ncut=30)

println("회로: 2 JJ (대칭) + 션트 C")
println("\n기호적 해밀토니안:")
println("  ", sym_hamiltonian(circ3))
println("\n외부 자속 변수: ", external_fluxes(circ3))

회로: 2 JJ (대칭) + 션트 C

기호적 해밀토니안:
  -10.0(cos(-Φext₁ + θ₁) + cos(-Φext₂ + θ₁)) + 1.8461538461538458(nθ₁^2)

외부 자속 변수: Symbolics.Num[Φext₁, Φext₂]


In [7]:
# 자속 스윕
println("자속에 따른 스펙트럼:")
println("  Φ_ext     ω₀₁ (GHz)    ω₁₂ (GHz)    α (GHz)")
println("  " * "-"^50)
for phi in range(0.0, π, length=11)
    set_external_flux!(circ3, 1, phi)
    e = eigenvals(circ3; evals_count=4)
    w01 = e[2] - e[1]
    w12 = e[3] - e[2]
    α = w12 - w01
    println("  $(round(phi, digits=3))     $(round(w01, digits=4))       $(round(w12, digits=4))       $(round(α, digits=4))")
end

자속에 따른 스펙트럼:
  Φ_ext     ω₀₁ (GHz)    ω₁₂ (GHz)    α (GHz)
  --------------------------------------------------
  0.0     8.1027       7.5646       -0.5381
  0.314     8.0494       7.5105       -0.5389
  0.628     7.8889       7.3476       -0.5412
  0.942     7.6188       7.0731       -0.5457
  1.257     7.2348       6.6812       -0.5535
  1.571     6.7289       6.161       -0.5679
  1.885     6.0873       5.489       -0.5984
  2.199     5.2849       4.6063       -0.6786
  2.513     4.2745       3.3516       -0.9229
  2.827     2.9871       1.5141       -1.4729
  3.142     1.8462       0.0       -1.8462


In [8]:
# 자속 스윕 플롯
set_external_flux!(circ3, 1, 0.0)
sd3 = get_spectrum_vs_paramvals(circ3, :flux, range(0.0, π, length=101); evals_count=4)

fig = Figure(size=(600, 400))
ax = Axis(fig[1, 1],
    xlabel="Φ_ext (rad)",
    ylabel="에너지 - E₀ (GHz)",
    title="대칭 SQUID 스펙트럼 vs 외부 자속")
for k in 1:4
    lines!(ax, collect(sd3.param_vals), sd3.eigenvalues[:, k] .- sd3.eigenvalues[:, 1])
end
fig

UndefVarError: UndefVarError: `Figure` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

---
## 예제 4: 비대칭 SQUID ($E_{J1} \neq E_{J2}$)

비대칭 SQUID에서는 frustration point ($\Phi_{\text{ext}} = \pi$)에서도 잔여 $E_J$가 존재합니다:

$$E_J^{\text{residual}} = |E_{J1} - E_{J2}|$$

비대칭도 파라미터: $d = (E_{J1} - E_{J2})/(E_{J1} + E_{J2})$

In [9]:
desc4 = """
branches:
  - [JJ, 0, 1, EJ=12.0, EC=0.3]
  - [JJ, 0, 1, EJ=8.0, EC=0.3]
  - [C, 0, 1, EC=0.5]
"""

circ4 = Circuit(desc4; ncut=30)

d_asym = (12.0 - 8.0) / (12.0 + 8.0)
println("비대칭 SQUID:")
println("  EJ1 = 12.0 GHz, EJ2 = 8.0 GHz")
println("  비대칭도 d = $d_asym")
println("  잔여 EJ (Φ=π): |EJ1 - EJ2| = $(abs(12.0 - 8.0)) GHz")

비대칭 SQUID:
  EJ1 = 12.0 GHz, EJ2 = 8.0 GHz
  비대칭도 d = 0.2
  잔여 EJ (Φ=π): |EJ1 - EJ2| = 4.0 GHz


In [10]:
# 대칭 vs 비대칭 비교 플롯
flux_range = range(0.0, π, length=101)
w01_sym = Float64[]
w01_asym = Float64[]

for phi in flux_range
    set_external_flux!(circ3, 1, phi)
    set_external_flux!(circ4, 1, phi)
    e_s = eigenvals(circ3; evals_count=2)
    e_a = eigenvals(circ4; evals_count=2)
    push!(w01_sym, e_s[2] - e_s[1])
    push!(w01_asym, e_a[2] - e_a[1])
end

fig = Figure(size=(600, 400))
ax = Axis(fig[1, 1],
    xlabel="Φ_ext (rad)",
    ylabel="ω₀₁ (GHz)",
    title="대칭 vs 비대칭 SQUID")
lines!(ax, collect(flux_range), w01_sym, label="대칭 (d=0)", linewidth=2)
lines!(ax, collect(flux_range), w01_asym, label="비대칭 (d=$(d_asym))", linewidth=2, linestyle=:dash)
axislegend(ax)
fig

UndefVarError: UndefVarError: `Figure` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

---
## 예제 5: 하드코딩 Transmon과의 검증

`Circuit`으로 구성한 Transmon의 고유값이 `Transmon` 타입의 결과와 일치하는지 검증합니다.

단일 JJ 브랜치의 $E_C$와 Transmon의 총 $E_C$의 관계:
$$E_{C,\text{total}} = 4 \times E_{C,\text{branch}}$$

따라서 $E_C = 1.2$ GHz를 얻으려면 $E_{C,\text{branch}} = 0.3$ GHz로 설정합니다.

In [11]:
desc_val = """
branches:
  - [JJ, 0, 1, EJ=30.0, EC=0.3]
"""
circ_val = Circuit(desc_val; ncut=30)
tmon_val = Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=30, truncated_dim=6)

e_circ = eigenvals(circ_val; evals_count=6)
e_tmon = eigenvals(tmon_val; evals_count=6)

println("  준위    Circuit         Transmon        차이")
println("  " * "-"^55)
for i in 1:6
    diff = abs(e_circ[i] - e_tmon[i])
    println("  E_$(i-1)     $(round(e_circ[i], digits=8))    $(round(e_tmon[i], digits=8))    $(diff)")
end

max_diff = maximum(abs.(e_circ .- e_tmon))
println("\n  최대 차이: $max_diff")
println("  검증 결과: ", max_diff < 1e-10 ? "통과" : "실패")

  준위    Circuit         Transmon        차이
  -------------------------------------------------------
  E_0     -21.82674091    -21.82674091    0.0
  E_1     -6.15959855    -6.15959855    0.0
  E_2     7.94124586    7.94124586    0.0
  E_3     20.84080224    20.84080224    0.0
  E_4     28.09898516    28.09898516    0.0
  E_5     45.79505843    45.79505843    0.0

  최대 차이: 0.0
  검증 결과: 통과
